# NVIDIA-Nemotron-3.5-Transcribe-30B-A4B (G4 / vLLM + NeMo Speech)

Serves `nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B` (BF16 / FP8 / NVFP4, 65536 context, up to 1-hour audio, 40 languages, MTP8 speculative decoding, CTC word timestamps & speaker diarization) on a Colab G4 GPU via `vllm serve` + `nemo_speechlm`, with Bearer token auth and direct Google Drive audio batch/single transcription endpoints.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "nemotron-transcribe")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- parallel NeMo Speech + vLLM 0.32 env & local SSD model download (67.5 GB BF16 / FP8 / NVFP4)
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import hashlib, platform

PRECISION_REPOS = {
    "bf16": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B",
    "fp8": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-FP8",
    "nvfp4": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-NVFP4",
}
PRECISION = os.environ.get("CT_NEMOTRON_PRECISION", "bf16").strip().lower()
MODEL_REPO = os.environ.get("CT_NEMOTRON_REPO", "").strip() or PRECISION_REPOS.get(PRECISION, PRECISION_REPOS["bf16"])
MODEL_ALIAS = os.environ.get("CT_NEMOTRON_ALIAS", "nemotron-transcribe").strip()
NEMO_REPO = os.environ.get("CT_NEMO_SPEECH_REPO", "https://github.com/NVIDIA-NeMo/Speech.git").strip()

NT_WORK = Path("/content/nemotron-transcribe")
NEMO_DIR = Path("/content/nemo-speech")
VENV_DIR = NEMO_DIR / ".venv"
MODEL_DIR = NT_WORK / "model" / MODEL_REPO.split("/")[-1]
DRIVE_AUDIO_DIR = Path(DATA_DIR) / "audio"
DRIVE_OUTPUT_DIR = Path(DATA_DIR) / "outputs"
FLASHINFER_DRIVE_DIR = Path(DATA_DIR) / "flashinfer-cache"
FLASHINFER_CACHE_DIR = NT_WORK / "flashinfer-cache"
TRITON_DRIVE_DIR = Path(DATA_DIR) / "triton-cache"
TRITON_CACHE_DIR = NT_WORK / "triton-cache"
VLLM_DRIVE_DIR = Path(DATA_DIR) / "vllm-cache"
VLLM_CACHE_DIR = NT_WORK / "vllm-cache"

for d in (
    NT_WORK, MODEL_DIR.parent, DRIVE_AUDIO_DIR, DRIVE_OUTPUT_DIR,
    FLASHINFER_DRIVE_DIR, FLASHINFER_CACHE_DIR,
    TRITON_DRIVE_DIR, TRITON_CACHE_DIR,
    VLLM_DRIVE_DIR, VLLM_CACHE_DIR,
):
    d.mkdir(parents=True, exist_ok=True)
def nlog(msg):
    print(time.strftime("[%F %T] [nemotron] ") + msg, flush=True)

# One archive per cache/environment/GPU avoids Drive's small-file latency.
(NT_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
sys.path.insert(0, str(NT_WORK))
from colab_cache import restore_cache, save_cache
CACHE_KEY = "vllm-0.32-py313-" + PRECISION + "-sm" + ct_gpu_stat("compute_cap").splitlines()[0].replace(".", "")
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_PAIRS = (
    (FLASHINFER_CACHE_DIR, FLASHINFER_DRIVE_DIR),
    (TRITON_CACHE_DIR, TRITON_DRIVE_DIR),
    (VLLM_CACHE_DIR, VLLM_DRIVE_DIR),
)
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        restore_cache(_local_c, _legacy_c, CACHE_ARCHIVES, CACHE_KEY, nlog)
    except (OSError, ValueError) as err:
        nlog(f"Cache restore warning ({_local_c.name}): {err}; continuing with local cache")

def run_logged(cmd, log_path, env=None, cwd=None, check=True):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True)
    if check and res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-40:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:4])}\n{tail}")
    return res

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], text=True
).strip()
nlog(f"GPU detected: {gpu_info} | repo={MODEL_REPO} precision={PRECISION}")

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg >/dev/null 2>&1", shell=True, check=False)

if not shutil.which("yt-dlp"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=False)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

def prepare_env():
    env_log = NT_WORK / "env-setup.log"
    marker = VENV_DIR / ".ct_env_ready"
    vllm_bin = VENV_DIR / "bin" / "vllm"
    py_bin = VENV_DIR / "bin" / "python"
    if marker.is_file() and vllm_bin.is_file() and py_bin.is_file():
        nlog("Reusing existing NeMo Speech + vLLM 0.32 virtualenv")
        return VENV_DIR

    nlog(f"Setting up NeMo Speech + vLLM 0.32 environment (log: {env_log})...")
    if not (NEMO_DIR / ".git").is_dir():
        run_logged(["git", "clone", "--depth", "1", NEMO_REPO, str(NEMO_DIR)], env_log)
    else:
        run_logged(["git", "-C", str(NEMO_DIR), "pull", "--ff-only"], env_log, check=False)

    uv_bin = shutil.which("uv") or "uv"
    run_logged([uv_bin, "venv", "--python", "3.13", str(VENV_DIR), "--clear"], env_log, cwd=str(NEMO_DIR))
    sync_env = {**os.environ, "UV_PROJECT_ENVIRONMENT": str(VENV_DIR)}
    res = run_logged(
        [uv_bin, "sync", "--extra", "asr", "--extra", "vllm", "--locked", "--no-install-project"],
        env_log, env=sync_env, cwd=str(NEMO_DIR), check=False
    )
    if res.returncode != 0:
        nlog("`uv sync --locked` exited non-zero; retrying without `--locked`...")
        run_logged(
            [uv_bin, "sync", "--extra", "asr", "--extra", "vllm", "--no-install-project"],
            env_log, env=sync_env, cwd=str(NEMO_DIR)
        )
    run_logged([uv_bin, "pip", "install", "--python", str(py_bin), "--no-deps", "-e", str(NEMO_DIR)], env_log)
    run_logged(
        [uv_bin, "pip", "install", "--python", str(py_bin), "--no-deps", "av==19.0.2", "opencv-python-headless==5.0.0.93"],
        env_log,
    )
    run_logged([uv_bin, "pip", "install", "--python", str(py_bin), "-U", "yt-dlp"], env_log, check=False)

    # Apply official NVIDIA DFlash2 safety patch to vLLM 0.32 if applicable
    patch_code = r"""
from pathlib import Path
import vllm
path = Path(vllm.__file__).parent / "model_executor/models/qwen3_dflash2.py"
if path.is_file():
    source = path.read_text()
    old = "    successors = successor_table[candidate_ids]\n"
    anchor = "            anchor_token_ids[:, None, None].expand(-1, 1, top_k),"
    if "safe_anchor_token_ids" not in source and source.count(old) == 1 and source.count(anchor) == 1:
        source = source.replace(old, (
            "    safe_anchor_token_ids = anchor_token_ids.clamp(\n"
            "        min=0, max=predecessor_table.shape[0] - 1\n"
            "    )\n" + old
        )).replace(anchor, "            safe_anchor_token_ids[:, None, None].expand(-1, 1, top_k),")
        path.write_text(source)
"""
    run_logged([str(py_bin), "-c", patch_code], env_log, check=False)
    run_logged([str(vllm_bin), "download-kernels"], env_log, env=sync_env, check=False)
    marker.write_text("ok\n", encoding="utf-8")
    nlog("NeMo Speech + vLLM 0.32 environment ready")
    return VENV_DIR

def _cleanup_hf_temp():
    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub", str(MODEL_DIR / ".cache")):
        shutil.rmtree(p, ignore_errors=True)

def prepare_model():
    dl_log = NT_WORK / "model-download.log"
    marker = MODEL_DIR / ".ct_model_ready"
    has_weights = any(MODEL_DIR.glob("*.safetensors")) if MODEL_DIR.is_dir() else False
    if marker.is_file() and (MODEL_DIR / "config.json").is_file() and has_weights:
        nlog(f"Reusing local SSD model at {MODEL_DIR}")
        return MODEL_DIR

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
    _cleanup_hf_temp()
    free_gb = shutil.disk_usage("/content").free / 1e9
    nlog(f"Downloading {MODEL_REPO} -> local SSD {MODEL_DIR} (free SSD: {free_gb:.1f} GB; log: {dl_log})...")
    dl_env = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    dl_env.pop("HF_XET_HIGH_PERFORMANCE", None)
    dl_code = (
        "import sys; from huggingface_hub import snapshot_download; "
        "snapshot_download(repo_id=sys.argv[1], local_dir=sys.argv[2])"
    )
    try:
        run_logged([sys.executable, "-c", dl_code, MODEL_REPO, str(MODEL_DIR)], dl_log, env=dl_env)
    finally:
        _cleanup_hf_temp()

    if not (MODEL_DIR / "config.json").is_file() or not any(MODEL_DIR.glob("*.safetensors")):
        raise RuntimeError(f"Model download incomplete in {MODEL_DIR}")
    marker.write_text(MODEL_REPO + "\n", encoding="utf-8")
    total_gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 1e9
    nlog(f"Model ready on local SSD: {MODEL_DIR} ({total_gb:.2f} GB)")
    return MODEL_DIR

with ThreadPoolExecutor(max_workers=2) as pool:
    f_env = pool.submit(prepare_env)
    f_mod = pool.submit(prepare_model)
    while not (f_env.done() and f_mod.done()):
        for fut in (f_env, f_mod):
            if fut.done() and fut.exception():
                raise fut.exception()
        free_gb = shutil.disk_usage("/content").free / 1e9
        nlog(f"Setup heartbeat | env={'done' if f_env.done() else 'running'} "
             f"model={'done' if f_mod.done() else 'running'} | free_ssd={free_gb:.1f} GB")
        time.sleep(30)
    VENV_DIR = f_env.result()
    MODEL_DIR = f_mod.result()

nlog(f"Setup complete: venv={VENV_DIR} model={MODEL_DIR} drive_data={DATA_DIR}")


In [ ]:
# @title 2. Service -- vLLM + NeMo Speech SALM on 127.0.0.1:8000 with Bearer Auth + Google Drive Audio API
(NT_WORK / "colab_asgi_common.py").write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(NT_WORK / "nemotron_audio.py").write_text('"""Shared audio chunking, SRT formatting, and prompt presets for Nemotron Transcribe."""\nfrom __future__ import annotations\n\nimport base64\nimport math\nimport re\nimport subprocess\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDIRECT_MIMES = {\n    ".wav": "audio/wav",\n    ".flac": "audio/flac",\n    ".mp3": "audio/mpeg",\n    ".ogg": "audio/ogg",\n    ".m4a": "audio/mp4",\n}\n\nPRESETS: dict[str, dict[str, Any]] = {\n    "verbatim": {\n        "description": "Verbatim transcription (keeps fillers, repetitions, punctuation and casing; recommended multilingual default)",\n        "user_prompt": (\n            "Audio transcription. Language: as spoken. Wording: verbatim, including fillers and repetitions. "\n            "Formatting: punctuation and capitalization. Output: transcript only."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "clean": {\n        "description": "Clean written transcription (removes fillers and accidental repetitions; keeps meaning, punctuation and casing)",\n        "user_prompt": (\n            "Audio transcription. Language: as spoken. Wording: omit fillers and accidental repetitions; "\n            "keep the remaining spoken wording unchanged. Formatting: punctuation and capitalization. "\n            "Output: transcript only."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "speaker": {\n        "description": "Speaker-diarized transcription + word-level timestamps (<spk:0> <spk:1> tags; use with timestamps=True)",\n        "user_prompt": (\n            "Transcribe this audio with punctuation and capitalization. Include all spoken words and verbal fillers. "\n            "Write numbers in their standard written form. Put speaker tags <spk:0> <spk:1> <spk:2> <spk:3> on the transcription."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": True,\n    },\n    "multispeaker": {\n        "description": "Multi-speaker plain-text transcription (no punctuation, lowercase, <spk:N> before each turn)",\n        "user_prompt": (\n            "Transcribe this audio. Include all spoken words and verbal fillers. Write numbers as spoken. "\n            "Use no punctuation and no capitalization. Start every speaker turn with a speaker tag such as "\n            "<spk:0>, <spk:1>, <spk:2>, etc.; assign one stable tag per speaker and keep speaker identities "\n            "consistent. Output only the speaker-tagged transcript."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "rich": {\n        "description": "Rich transcription (includes non-speech events <ev:*> and speaking-style <sty:*> tags)",\n        "user_prompt": (\n            "Transcribe this audio with punctuation and capitalization. Include all spoken words and verbal fillers. "\n            "Write numbers in their standard written form. Use <ev:*> and <sty:*> special tokens to recognize all "\n            "non-speech verbatims and speaking style."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "near-field": {\n        "description": "Transcribe only the speaker closest to the microphone (filters far-field background voices)",\n        "user_prompt": (\n            "Transcribe this audio. Include all spoken words and verbal fillers. Write numbers as spoken. "\n            "No punctuation or capitalization. Transcribe only the speaker nearest to the microphone. "\n            "Do not transcribe any other voices."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "structured": {\n        "description": "Structured-content transcription (precisely keeps emails, URLs, file paths, commands, numbers and punctuation separators)",\n        "user_prompt": (\n            "Transcribe as a clean written transcript. Preserve exact values and punctuation/separators in emails, "\n            "URLs, file paths, commands, and identifiers. Use conventional written forms."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "context": {\n        "description": "Context-biased transcription with domain vocabulary / proper-noun hints (use with the context parameter)",\n        "user_prompt": "Transcribe with punctuation and capitalization. Context terms: {context}.",\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "translate": {\n        "description": "Direct speech translation into target-language text (use with the lang parameter, e.g. Chinese / English / Japanese)",\n        "user_prompt": "Please translate the given speech to {lang}.",\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "lid": {\n        "description": "Language identification (outputs the language label of the recording)",\n        "user_prompt": (\n            "Identify the language spoken in this audio. Respond with exactly one of the following locale labels "\n            "and nothing else: ar_eg, bg_bg, cmn_hans_cn, hr_hr, cs_cz, da_dk, nl_nl, en_us, et_ee, fil_ph, fi_fi, "\n            "fr_fr, de_de, el_gr, he_il, hi_in, hu_hu, id_id, it_it, ja_jp, ko_kr, lv_lv, lt_lt, mt_mt, pl_pl, "\n            "pt_br, ro_ro, ru_ru, sk_sk, sl_si, es_419, sv_se, th_th, tr_tr, uk_ua, vi_vn."\n        ),\n        "system_prompt": "",\n        "reasoning": False,\n        "timestamps": False,\n    },\n    "qa": {\n        "description": "Audio understanding and Q&A (reasoning mode enable_thinking=True; answers questions about the recording)",\n        "user_prompt": "What is happening in this audio?",\n        "system_prompt": (\n            "Answer the question directly using evidence from every provided audio clip in order. "\n            "Answer every requested part, preserve exact names and numbers, and give a concise, specific answer first. "\n            "Do not invent details."\n        ),\n        "reasoning": True,\n        "timestamps": False,\n    },\n}\n\nYTDLP_DOMAIN_MARKERS = (\n    "youtube.com/", "youtu.be/", "twitter.com/", "x.com/", "t.co/",\n    "bilibili.com/", "b23.tv/", "tiktok.com/", "soundcloud.com/",\n    "vimeo.com/", "twitch.tv/", "xiaoyuzhoufm.com/", "podcasts.apple.com/",\n    "ytdl://", "yt-dlp://",\n)\n\nCHAR_RUN = re.compile(r"(.)\\1{20,}", re.DOTALL)\nBLOCK_RUN = re.compile(r"(.{1,4096}?)\\1{19,}", re.DOTALL)\n\n\ndef cleanup_generation_loops(text: str) -> str:\n    if not text:\n        return text\n    text = CHAR_RUN.sub(lambda m: m.group(1), text)\n    return BLOCK_RUN.sub(lambda m: m.group(1), text)\n\n\ndef probe_audio_duration_sec(audio_path: Path) -> float:\n    """Return audio/video duration in seconds using ffprobe (0.0 if unknown)."""\n    try:\n        res = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-show_entries", "format=duration",\n                "-of", "default=noprint_wrappers=1:nokey=1",\n                str(audio_path),\n            ],\n            stdout=subprocess.PIPE,\n            stderr=subprocess.PIPE,\n            text=True,\n            timeout=30,\n            check=False,\n        )\n        val = (res.stdout or "").strip()\n        return float(val) if val and val != "N/A" else 0.0\n    except Exception:\n        return 0.0\n\n\ndef find_silence_near(audio_path: Path, target_sec: float, window_sec: float = 45.0) -> float:\n    """Search for a natural speech pause within [target_sec - window_sec, target_sec + window_sec]."""\n    win_start = max(0.0, target_sec - window_sec)\n    win_dur = window_sec * 2.0\n    try:\n        res = subprocess.run(\n            [\n                "ffmpeg", "-nostdin", "-hide_banner",\n                "-ss", f"{win_start:.3f}", "-t", f"{win_dur:.3f}",\n                "-i", str(audio_path),\n                "-vn", "-af", "silencedetect=n=-35dB:d=0.35",\n                "-f", "null", "-",\n            ],\n            stdout=subprocess.PIPE,\n            stderr=subprocess.PIPE,\n            text=True,\n            timeout=25,\n            check=False,\n        )\n        log_out = res.stderr or ""\n        starts = [float(x) for x in re.findall(r"silence_start:\\s*([0-9.]+)", log_out)]\n        ends = [float(x) for x in re.findall(r"silence_end:\\s*([0-9.]+)", log_out)]\n        candidates: list[float] = []\n        for idx, st in enumerate(starts):\n            ed = ends[idx] if idx < len(ends) else (st + 0.35)\n            mid = win_start + (st + ed) / 2.0\n            candidates.append(mid)\n        if candidates:\n            return min(candidates, key=lambda c: abs(c - target_sec))\n    except Exception:\n        pass\n    return target_sec\n\n\ndef compute_chunk_ranges(\n    audio_path: Path,\n    total_duration: float,\n    max_chunk_sec: float = 2400.0,\n) -> list[tuple[float, float]]:\n    """Split audio longer than max_chunk_sec into balanced, silence-aligned (start_sec, end_sec) chunks."""\n    if max_chunk_sec <= 0 or total_duration <= 0 or total_duration <= max_chunk_sec:\n        return [(0.0, max(total_duration, 0.0))]\n\n    num_chunks = max(2, int(math.ceil(total_duration / max_chunk_sec)))\n    ideal_len = total_duration / num_chunks\n    boundaries = [0.0]\n    for k in range(1, num_chunks):\n        target = k * ideal_len\n        split_pt = find_silence_near(audio_path, target, window_sec=min(45.0, ideal_len * 0.15))\n        split_pt = max(boundaries[-1] + 30.0, min(split_pt, total_duration - 30.0))\n        boundaries.append(round(split_pt, 3))\n    boundaries.append(round(total_duration, 3))\n    return [(boundaries[i], boundaries[i + 1]) for i in range(len(boundaries) - 1)]\n\n\ndef load_audio_as_data_url(\n    audio_path: Path,\n    *,\n    force_ffmpeg_16k: bool = False,\n    start_sec: float | None = None,\n    end_sec: float | None = None,\n    direct_max_bytes: int = 140 * 1024 * 1024,\n) -> str:\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    ext = audio_path.suffix.lower()\n    sz = audio_path.stat().st_size\n    is_slice = (start_sec is not None and start_sec > 0.0) or (end_sec is not None and end_sec > 0.0)\n    if not is_slice and not force_ffmpeg_16k and ext in DIRECT_MIMES and sz <= direct_max_bytes:\n        b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n        return f"data:{DIRECT_MIMES[ext]};base64,{b64}"\n    cmd = ["ffmpeg", "-nostdin", "-v", "error"]\n    if start_sec is not None and start_sec > 0.0:\n        cmd.extend(["-ss", f"{start_sec:.3f}"])\n    if end_sec is not None and end_sec > 0.0:\n        dur = max(0.1, end_sec - (start_sec or 0.0))\n        cmd.extend(["-t", f"{dur:.3f}"])\n    cmd.extend([\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ])\n    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=300, check=False)\n    if res.returncode != 0 or not res.stdout:\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef shift_ctc_timestamps(ctc_ts: dict[str, Any] | None, offset_sec: float) -> dict[str, Any] | None:\n    if not isinstance(ctc_ts, dict) or offset_sec <= 0.0:\n        return ctc_ts\n    shifted: dict[str, Any] = dict(ctc_ts)\n    for key in ("words", "segments", "diarization"):\n        items = ctc_ts.get(key)\n        if isinstance(items, list):\n            new_items = []\n            for item in items:\n                if isinstance(item, dict):\n                    cp = dict(item)\n                    if cp.get("start") is not None:\n                        cp["start"] = round(float(cp["start"]) + offset_sec, 3)\n                    if cp.get("end") is not None:\n                        cp["end"] = round(float(cp["end"]) + offset_sec, 3)\n                    new_items.append(cp)\n                else:\n                    new_items.append(item)\n            shifted[key] = new_items\n    return shifted\n\n\ndef fmt_srt_time(seconds: float) -> str:\n    ms = max(0, int(round(seconds * 1000)))\n    h, rem = divmod(ms, 3600_000)\n    m, rem = divmod(rem, 60_000)\n    s, ms = divmod(rem, 1000)\n    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"\n\n\ndef words_to_srt(words: list[dict[str, Any]]) -> str:\n    if not words:\n        return ""\n    segments = []\n    cur_words: list[str] = []\n    cur_start = 0.0\n    cur_end = 0.0\n    cur_spk = None\n\n    def flush():\n        nonlocal cur_words, cur_start, cur_end, cur_spk\n        if not cur_words:\n            return\n        prefix = f"[{cur_spk}] " if cur_spk else ""\n        text = prefix + " ".join(cur_words).strip()\n        segments.append((cur_start, max(cur_end, cur_start + 0.2), text))\n        cur_words = []\n\n    for w in words:\n        w_text = str(w.get("word") or "").strip()\n        if not w_text:\n            continue\n        w_start = float(w.get("start") or 0.0)\n        w_end = float(w.get("end") or w_start)\n        w_spk = w.get("speaker")\n        if not cur_words:\n            cur_start = w_start\n            cur_end = w_end\n            cur_spk = w_spk\n            cur_words.append(w_text)\n            continue\n        gap = w_start - cur_end\n        spk_changed = w_spk != cur_spk\n        if (\n            spk_changed\n            or gap > 1.2\n            or len(cur_words) >= 14\n            or (len(cur_words) >= 6 and cur_words[-1].endswith((".", "?", "!", "\\u3002", "\\uff1f", "\\uff01")))\n        ):\n            flush()\n            cur_start = w_start\n            cur_end = w_end\n            cur_spk = w_spk\n            cur_words.append(w_text)\n        else:\n            cur_end = max(cur_end, w_end)\n            cur_words.append(w_text)\n    flush()\n\n    lines = []\n    for idx, (st_t, ed_t, txt) in enumerate(segments, 1):\n        lines.append(f"{idx}\\n{fmt_srt_time(st_t)} --> {fmt_srt_time(ed_t)}\\n{txt}\\n")\n    return "\\n".join(lines)\n\n\ndef build_prompt_and_settings(body: dict[str, Any]) -> tuple[str, str, bool, bool]:\n    preset_name = (body.get("preset") or "").strip().lower()\n    preset = PRESETS.get(preset_name) if preset_name else None\n\n    user_prompt = body.get("prompt")\n    if user_prompt is None or str(user_prompt).strip() == "":\n        user_prompt = preset["user_prompt"] if preset else PRESETS["verbatim"]["user_prompt"]\n    user_prompt = str(user_prompt)\n\n    context_terms = (body.get("context") or "").strip()\n    target_lang = (body.get("lang") or body.get("language") or "Chinese").strip()\n    if "{context}" in user_prompt:\n        user_prompt = user_prompt.replace("{context}", context_terms or "general")\n    elif context_terms and "Context terms:" not in user_prompt:\n        user_prompt = f"{user_prompt.rstrip()} Context terms: {context_terms}."\n    if "{lang}" in user_prompt:\n        user_prompt = user_prompt.replace("{lang}", target_lang)\n\n    sys_prompt = body.get("system_prompt")\n    if sys_prompt is None:\n        sys_prompt = preset["system_prompt"] if preset else ""\n    sys_prompt = str(sys_prompt)\n\n    reasoning = body.get("reasoning")\n    if reasoning is None:\n        reasoning = bool(preset["reasoning"]) if preset else False\n    else:\n        reasoning = bool(reasoning)\n\n    timestamps = body.get("timestamps")\n    if timestamps is None:\n        timestamps = bool(preset["timestamps"]) if preset else False\n    else:\n        timestamps = bool(timestamps)\n\n    return user_prompt, sys_prompt, reasoning, timestamps\n', encoding="utf-8")
(NT_WORK / "ct_nemotron_middleware.py").write_text('"""Authenticated ASGI middleware and Google Drive / yt-dlp endpoints for Nemotron Transcribe."""\nfrom __future__ import annotations\n\nimport asyncio\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nimport uuid\nfrom http import HTTPStatus\nfrom pathlib import Path\nfrom typing import Any\n\nfrom colab_asgi_common import (\n    AUDIO_EXTS,\n    list_drive_audio_files,\n    rel_to_mydrive,\n    resolve_drive_path,\n    verify_bearer_auth,\n)\nfrom nemotron_audio import (\n    PRESETS,\n    YTDLP_DOMAIN_MARKERS,\n    build_prompt_and_settings,\n    cleanup_generation_loops,\n    compute_chunk_ranges,\n    fmt_srt_time,\n    load_audio_as_data_url,\n    probe_audio_duration_sec,\n    shift_ctc_timestamps,\n    words_to_srt,\n)\n\ntry:\n    from starlette.responses import JSONResponse\nexcept ImportError:  # pragma: no cover - local unit-test fallback without starlette\n    JSONResponse = None  # type: ignore[assignment,misc]\n\ntry:\n    from nemo.collections.speechlm2.vllm.salm.ctc_serving import ctc_timestamp_middleware\nexcept ImportError:  # pragma: no cover - local unit-test fallback without nemo\n    ctc_timestamp_middleware = None  # type: ignore[assignment]\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nSERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR", "/content/drive/MyDrive/colab-llm/nemotron-transcribe"))\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = Path("/content/drive/MyDrive")\nMODEL_REPO = os.environ.get("CT_NEMOTRON_REPO", "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B")\n\n\ndef _check_auth(request: Any) -> bool:\n    return verify_bearer_auth(request.headers, API_TOKEN)\n\n\ndef _resolve_path(raw_spec: str, default_subdir: str | None = "audio") -> Path:\n    return resolve_drive_path(\n        raw_spec,\n        data_dir=DATA_DIR,\n        default_subdir=default_subdir,\n        mydrive_root=MYDRIVE_ROOT,\n        drive_root=DRIVE_ROOT,\n    )\n\n\ndef _call_local_chat_completions(payload: dict[str, Any], timeout: int = 1800) -> tuple[int, dict[str, Any]]:\n    headers = {"Content-Type": "application/json"}\n    if API_TOKEN:\n        headers["Authorization"] = f"Bearer {API_TOKEN}"\n    req = urllib.request.Request(\n        f"http://127.0.0.1:{SERVICE_PORT}/v1/chat/completions",\n        data=json.dumps(payload).encode("utf-8"),\n        headers=headers,\n        method="POST",\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=timeout) as resp:\n            return resp.status, json.loads(resp.read().decode("utf-8"))\n    except urllib.error.HTTPError as e:\n        raw = e.read().decode("utf-8", errors="replace")\n        try:\n            return e.code, json.loads(raw)\n        except Exception:\n            return e.code, {"error": {"message": raw, "code": e.code}}\n\n\ndef _is_media_url(spec: str) -> bool:\n    s = (spec or "").strip()\n    return s.startswith(("http://", "https://", "ytdl://", "yt-dlp://"))\n\n\ndef _clean_media_url(spec: str) -> str:\n    s = (spec or "").strip()\n    for pfx in ("ytdl://", "yt-dlp://"):\n        if s.startswith(pfx):\n            return s[len(pfx):]\n    return s\n\n\ndef _ensure_ytdlp() -> str:\n    for cand in (\n        shutil.which("yt-dlp"),\n        "/content/nemo-speech/.venv/bin/yt-dlp",\n        "/usr/local/bin/yt-dlp",\n    ):\n        if cand and os.path.isfile(cand):\n            return cand\n    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=True)\n    return shutil.which("yt-dlp") or "yt-dlp"\n\n\ndef _prepare_cookies_file(cookies_spec: str = "") -> Path | None:\n    candidates: list[Path] = []\n    if cookies_spec:\n        candidates.append(_resolve_path(cookies_spec, default_subdir=None))\n    candidates.extend([\n        DATA_DIR / "cookies.txt",\n        MYDRIVE_ROOT / "colab-llm" / "cookies.txt",\n        MYDRIVE_ROOT / "cookies.txt",\n    ])\n    for c in candidates:\n        if c.is_file() and c.stat().st_size > 0:\n            tmp_cookie = Path(f"/tmp/ct_ytdlp_cookies_{os.getpid()}_{uuid.uuid4().hex[:8]}.txt")\n            tmp_cookie.touch(mode=0o600)\n            shutil.copyfile(c, tmp_cookie)\n            return tmp_cookie\n    return None\n\n\ndef _sanitize_filename(name: str, max_len: int = 80) -> str:\n    cleaned = re.sub(r\'[\\\\/:*?"<>|\\r\\n\\t]+\', "_", (name or "").strip())\n    cleaned = re.sub(r"\\s+", "_", cleaned).strip("._ ")\n    return (cleaned[:max_len] or "audio").strip("._ ")\n\n\ndef _download_media_url_to_drive(\n    raw_url: str,\n    download_dir: str = "",\n    audio_format: str = "mp3",\n    cookies_drive: str = "",\n    playlist: bool = False,\n    force_download: bool = False,\n    download_video: bool = False,\n    save_media_to_drive: bool = True,\n) -> list[dict[str, Any]]:\n    url = _clean_media_url(raw_url)\n    if not url.startswith(("http://", "https://")):\n        raise ValueError(f"Unsupported media URL scheme: {raw_url!r}")\n    if save_media_to_drive:\n        target_dir = _resolve_path(download_dir, default_subdir=None) if (download_dir and download_dir.strip()) else (DATA_DIR / "audio" / "yt-dlp")\n    else:\n        target_dir = Path("/content/nemotron-transcribe/ytdlp-temp") / uuid.uuid4().hex[:12]\n    target_dir.mkdir(parents=True, exist_ok=True)\n    fmt = (audio_format or "mp3").strip().lower().lstrip(".")\n    if fmt not in ("mp3", "m4a", "wav", "opus", "flac"):\n        fmt = "mp3"\n\n    ytdlp_bin = _ensure_ytdlp()\n    stage_dir = Path("/content/nemotron-transcribe/ytdlp-stage") / uuid.uuid4().hex[:12]\n    stage_dir.mkdir(parents=True, exist_ok=True)\n    cookie_file = _prepare_cookies_file(cookies_drive)\n\n    try:\n        out_tmpl = str(stage_dir / "%(extractor)s__%(id)s.%(ext)s")\n        cmd = [\n            ytdlp_bin,\n            "--no-progress",\n            "--no-warnings",\n        ]\n        if download_video:\n            cmd.extend([\n                "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/bv*+ba/b",\n                "--merge-output-format", "mp4",\n            ])\n        else:\n            cmd.extend([\n                "--extract-audio",\n                "--audio-format", fmt,\n                "--audio-quality", "0",\n            ])\n        cmd.extend([\n            "--write-info-json",\n            "--output", out_tmpl,\n        ])\n        if not playlist:\n            cmd.append("--no-playlist")\n        else:\n            cmd.append("--yes-playlist")\n        if cookie_file:\n            cmd.extend(["--cookies", str(cookie_file)])\n        cmd.append(url)\n\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=1800, check=False)\n        info_files = sorted(stage_dir.glob("*.info.json"))\n        if res.returncode != 0 and not info_files:\n            err = (res.stderr or res.stdout or "").strip()[-1200:]\n            raise RuntimeError(f"yt-dlp failed for {url}: {err}")\n\n        downloaded_items: list[dict[str, Any]] = []\n        for info_path in info_files:\n            try:\n                info = json.loads(info_path.read_text(encoding="utf-8", errors="replace"))\n            except Exception:\n                info = {}\n            if info.get("_type") == "playlist":\n                continue\n            stem_prefix = info_path.name[:-len(".info.json")]\n            audio_candidates = [\n                p for p in stage_dir.glob(f"{stem_prefix}.*")\n                if p.suffix.lower() in AUDIO_EXTS and not p.name.endswith(".info.json")\n            ]\n            if not audio_candidates:\n                continue\n            staged_audio = audio_candidates[0]\n            extractor = _sanitize_filename(str(info.get("extractor") or info.get("extractor_key") or "media"), 24).lower()\n            vid_id = _sanitize_filename(str(info.get("id") or uuid.uuid4().hex[:8]), 40)\n            title = _sanitize_filename(str(info.get("title") or vid_id), 70)\n            dest_name = f"{extractor}_{vid_id}_{title}{staged_audio.suffix.lower()}"\n            dest_path = target_dir / dest_name\n\n            if save_media_to_drive and not force_download and not dest_path.is_file():\n                existing = [\n                    p for p in target_dir.glob(f"{extractor}_{vid_id}_*{staged_audio.suffix.lower()}")\n                    if p.is_file() and p.stat().st_size > 0\n                ]\n                if existing:\n                    dest_path = existing[0]\n\n            if force_download or not dest_path.is_file() or dest_path.stat().st_size == 0:\n                partial_path = target_dir / f".{dest_name}.{os.getpid()}.partial"\n                shutil.copyfile(staged_audio, partial_path)\n                partial_path.replace(dest_path)\n\n            st = dest_path.stat()\n            downloaded_items.append({\n                "source_url": raw_url,\n                "webpage_url": info.get("webpage_url") or url,\n                "video_id": info.get("id"),\n                "extractor": info.get("extractor_key") or info.get("extractor"),\n                "title": info.get("title"),\n                "uploader": info.get("uploader"),\n                "duration": info.get("duration"),\n                "media_type": "video" if download_video else "audio",\n                "saved_to_drive": save_media_to_drive,\n                "drive_path": str(dest_path),\n                "rel_path": rel_to_mydrive(dest_path, MYDRIVE_ROOT) if save_media_to_drive else dest_path.name,\n                "name": dest_path.name,\n                "size_bytes": st.st_size,\n                "size_mb": round(st.st_size / 1e6, 2),\n            })\n\n        if not downloaded_items:\n            for p in sorted(stage_dir.iterdir()):\n                if p.is_file() and p.suffix.lower() in AUDIO_EXTS:\n                    dest_path = target_dir / _sanitize_filename(p.stem, 90)\n                    dest_path = dest_path.with_suffix(p.suffix.lower())\n                    shutil.copyfile(p, dest_path)\n                    st = dest_path.stat()\n                    downloaded_items.append({\n                        "source_url": raw_url,\n                        "webpage_url": url,\n                        "media_type": "video" if download_video else "audio",\n                        "saved_to_drive": save_media_to_drive,\n                        "drive_path": str(dest_path),\n                        "rel_path": rel_to_mydrive(dest_path, MYDRIVE_ROOT) if save_media_to_drive else dest_path.name,\n                        "name": dest_path.name,\n                        "size_bytes": st.st_size,\n                        "size_mb": round(st.st_size / 1e6, 2),\n                    })\n\n        if not downloaded_items:\n            err = (res.stderr or res.stdout or "").strip()[-1200:]\n            raise RuntimeError(f"yt-dlp did not produce any media files for {url}. Output: {err}")\n        return downloaded_items\n    finally:\n        shutil.rmtree(stage_dir, ignore_errors=True)\n        if cookie_file and cookie_file.is_file():\n            cookie_file.unlink(missing_ok=True)\n\n\ndef _run_drive_transcribe_sync(body: dict[str, Any]) -> tuple[int, dict[str, Any]]:\n    t0 = time.monotonic()\n    raw_path = (\n        body.get("drive_path") or body.get("audio") or body.get("path") or body.get("url") or body.get("media_url") or ""\n    ).strip()\n    if not raw_path:\n        return 400, {"error": "Missing \'drive_path\' or \'url\' (Google Drive audio file path or media URL)"}\n\n    media_meta: dict[str, Any] | None = None\n    save_media_to_drive = bool(body.get("save_media_to_drive", True))\n    temp_media_path: Path | None = None\n    try:\n        if _is_media_url(raw_path):\n            dl_items = _download_media_url_to_drive(\n                raw_url=raw_path,\n                download_dir=str(body.get("download_dir") or ""),\n                audio_format=str(body.get("audio_format") or "mp3"),\n                cookies_drive=str(body.get("cookies_drive") or ""),\n                playlist=bool(body.get("playlist", False)),\n                force_download=bool(body.get("force_download", False)),\n                download_video=bool(body.get("download_video", False)),\n                save_media_to_drive=save_media_to_drive,\n            )\n            media_meta = dl_items[0]\n            audio_path = Path(media_meta["drive_path"])\n            if not save_media_to_drive:\n                temp_media_path = audio_path\n        else:\n            audio_path = _resolve_path(raw_path)\n    except ValueError as err:\n        return 400, {"error": str(err)}\n\n    try:\n        if not audio_path.is_file():\n            return 404, {\n                "error": f"Audio file not found on Google Drive: \'{raw_path}\' (resolved to \'{audio_path}\')",\n                "resolved_path": str(audio_path),\n                "drive_mounted": MYDRIVE_ROOT.is_dir(),\n            }\n\n        user_prompt, sys_prompt, reasoning, timestamps = build_prompt_and_settings(body)\n        force_ffmpeg = bool(body.get("normalize_audio", False))\n\n        total_duration = probe_audio_duration_sec(audio_path)\n        raw_chunk_min = body.get("chunk_minutes")\n        if raw_chunk_min is not None:\n            max_chunk_sec = float(raw_chunk_min) * 60.0\n        elif body.get("chunk_seconds") is not None:\n            max_chunk_sec = float(body["chunk_seconds"])\n        else:\n            max_chunk_sec = 2400.0\n\n        chunk_ranges = compute_chunk_ranges(audio_path, total_duration, max_chunk_sec=max_chunk_sec)\n        is_multi_chunk = len(chunk_ranges) > 1\n\n        extra_body: dict[str, Any] = {\n            "chat_template_kwargs": {"enable_thinking": reasoning},\n            "skip_special_tokens": bool(body.get("skip_special_tokens", False)),\n            "spaces_between_special_tokens": False,\n            "seed": int(body.get("seed", 0)),\n        }\n        if reasoning:\n            extra_body.update(top_k=-1, thinking_token_budget=int(body.get("thinking_token_budget", 17408)))\n            extra_body["chat_template_kwargs"]["reasoning_budget"] = int(body.get("reasoning_budget", 16384))\n        if timestamps:\n            extra_body["mm_processor_kwargs"] = {"capture_ctc_timestamps": True}\n\n        temp = body.get("temperature")\n        if temp is None:\n            temp = 0.6 if reasoning else 0.0\n        top_p = body.get("top_p")\n        if top_p is None:\n            top_p = 0.95 if reasoning else 1.0\n        max_tokens = body.get("max_tokens")\n        if max_tokens is None:\n            max_tokens = 20480 if reasoning else 16384\n\n        texts: list[str] = []\n        reasonings: list[str] = []\n        merged_words: list[dict[str, Any]] = []\n        merged_segments: list[dict[str, Any]] = []\n        merged_diarization: list[dict[str, Any]] = []\n        last_ctc_ts: dict[str, Any] | None = None\n        chunk_meta_list: list[dict[str, Any]] = []\n        total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}\n\n        for c_idx, (c_start, c_end) in enumerate(chunk_ranges, 1):\n            c_t0 = time.monotonic()\n            if is_multi_chunk:\n                data_url = load_audio_as_data_url(audio_path, force_ffmpeg_16k=True, start_sec=c_start, end_sec=c_end)\n            else:\n                data_url = load_audio_as_data_url(audio_path, force_ffmpeg_16k=force_ffmpeg)\n\n            messages: list[dict[str, Any]] = []\n            if sys_prompt:\n                messages.append({"role": "system", "content": sys_prompt})\n            messages.append({\n                "role": "user",\n                "content": [\n                    {"type": "text", "text": user_prompt},\n                    {"type": "audio_url", "audio_url": {"url": data_url}},\n                ],\n            })\n\n            chat_payload = {\n                "model": body.get("model") or MODEL_REPO,\n                "messages": messages,\n                "temperature": float(temp),\n                "top_p": float(top_p),\n                "max_tokens": int(max_tokens),\n                **extra_body,\n            }\n\n            status_code, completion = _call_local_chat_completions(chat_payload)\n            if status_code != 200:\n                return status_code, completion\n\n            choice = (completion.get("choices") or [{}])[0]\n            msg = choice.get("message") or {}\n            c_text = (msg.get("content") or "").strip()\n            c_reasoning = msg.get("reasoning_content") or msg.get("reasoning") or None\n            if not reasoning and body.get("cleanup_loops", True):\n                c_text = cleanup_generation_loops(c_text)\n\n            if c_text:\n                if is_multi_chunk and reasoning:\n                    texts.append(f"[{fmt_srt_time(c_start)[:8]} - {fmt_srt_time(c_end)[:8]}]\\n{c_text}")\n                else:\n                    texts.append(c_text)\n            if c_reasoning:\n                if is_multi_chunk:\n                    reasonings.append(f"[Chunk {c_idx}/{len(chunk_ranges)}: {fmt_srt_time(c_start)[:8]} - {fmt_srt_time(c_end)[:8]}]\\n{c_reasoning}")\n                else:\n                    reasonings.append(str(c_reasoning))\n\n            c_ctc = shift_ctc_timestamps(completion.get("ctc_timestamps"), c_start)\n            if isinstance(c_ctc, dict):\n                last_ctc_ts = c_ctc\n                if isinstance(c_ctc.get("words"), list):\n                    merged_words.extend(c_ctc["words"])\n                if isinstance(c_ctc.get("segments"), list):\n                    merged_segments.extend(c_ctc["segments"])\n                if isinstance(c_ctc.get("diarization"), list):\n                    merged_diarization.extend(c_ctc["diarization"])\n\n            u = completion.get("usage")\n            if isinstance(u, dict):\n                for k in ("prompt_tokens", "completion_tokens", "total_tokens"):\n                    total_usage[k] += int(u.get(k) or 0)\n\n            chunk_meta_list.append({\n                "chunk": c_idx,\n                "start_sec": round(c_start, 2),\n                "end_sec": round(c_end, 2),\n                "duration_sec": round(max(0.0, c_end - c_start), 2),\n                "elapsed_sec": round(time.monotonic() - c_t0, 2),\n            })\n\n        text = ("\\n\\n" if is_multi_chunk else "").join(texts)\n        reasoning_content = "\\n\\n".join(reasonings) if reasonings else None\n\n        if last_ctc_ts is not None:\n            ctc_ts: dict[str, Any] | None = dict(last_ctc_ts)\n            if merged_words:\n                ctc_ts["words"] = merged_words\n            if merged_segments:\n                ctc_ts["segments"] = merged_segments\n            if merged_diarization:\n                ctc_ts["diarization"] = merged_diarization\n        else:\n            ctc_ts = None\n\n        srt_text = words_to_srt(merged_words) if merged_words else ""\n\n        elapsed = round(time.monotonic() - t0, 2)\n        result: dict[str, Any] = {\n            "drive_path": str(audio_path),\n            "rel_path": rel_to_mydrive(audio_path, MYDRIVE_ROOT),\n            "source_url": media_meta["source_url"] if media_meta else None,\n            "media_info": media_meta,\n            "duration_sec": round(total_duration, 2) if total_duration > 0 else None,\n            "chunk_count": len(chunk_ranges),\n            "chunks": chunk_meta_list if is_multi_chunk else None,\n            "prompt": user_prompt,\n            "system_prompt": sys_prompt,\n            "preset": body.get("preset") or None,\n            "reasoning": reasoning,\n            "timestamps": timestamps,\n            "text": text,\n            "reasoning_content": reasoning_content,\n            "ctc_timestamps": ctc_ts,\n            "srt": srt_text or None,\n            "elapsed_sec": elapsed,\n            "usage": total_usage if total_usage["total_tokens"] > 0 else None,\n            "saved_drive_files": [],\n        }\n\n        save_to_drive = bool(body.get("save_to_drive", False))\n        save_drive_dir = (body.get("save_drive_dir") or "").strip()\n        if save_to_drive or save_drive_dir:\n            if save_drive_dir:\n                out_dir = _resolve_path(save_drive_dir, default_subdir=None)\n            elif temp_media_path is not None:\n                out_dir = DATA_DIR / "transcripts"\n            else:\n                out_dir = audio_path.parent\n            out_dir.mkdir(parents=True, exist_ok=True)\n            stem = audio_path.stem\n            txt_file = out_dir / f"{stem}.txt"\n            txt_file.write_text(text + ("\\n" if not text.endswith("\\n") else ""), encoding="utf-8")\n            result["saved_drive_files"].append(str(txt_file))\n\n            json_file = out_dir / f"{stem}.nemotron.json"\n            json_file.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n            result["saved_drive_files"].append(str(json_file))\n\n            if srt_text:\n                srt_file = out_dir / f"{stem}.srt"\n                srt_file.write_text(srt_text, encoding="utf-8")\n                result["saved_drive_files"].append(str(srt_file))\n\n        if temp_media_path is not None:\n            result["drive_path"] = None\n\n        return 200, result\n    except ValueError as err:\n        return 400, {"error": str(err)}\n    finally:\n        if temp_media_path is not None:\n            shutil.rmtree(temp_media_path.parent, ignore_errors=True)\n\n\ndef _rewrite_drive_urls_in_chat_body(raw_body: bytes) -> bytes | None:\n    if (\n        b"drive://" not in raw_body\n        and b"gdrive://" not in raw_body\n        and b"file://" not in raw_body\n        and b"/content/" not in raw_body\n        and not any(m.encode() in raw_body for m in YTDLP_DOMAIN_MARKERS)\n    ):\n        return None\n    try:\n        payload = json.loads(raw_body)\n    except Exception:\n        return None\n    if not isinstance(payload, dict):\n        return None\n    changed = False\n    for msg in payload.get("messages") or []:\n        content = msg.get("content") if isinstance(msg, dict) else None\n        if not isinstance(content, list):\n            continue\n        for item in content:\n            if not isinstance(item, dict) or item.get("type") != "audio_url":\n                continue\n            aurl = item.get("audio_url")\n            url_val = aurl.get("url") if isinstance(aurl, dict) else None\n            if not isinstance(url_val, str):\n                continue\n            if url_val.startswith(("drive://", "gdrive://", "file://", "/content/")):\n                ap = _resolve_path(url_val)\n                aurl["url"] = load_audio_as_data_url(ap)\n                changed = True\n            elif any(m in url_val for m in YTDLP_DOMAIN_MARKERS):\n                dl_items = _download_media_url_to_drive(url_val)\n                ap = Path(dl_items[0]["drive_path"])\n                aurl["url"] = load_audio_as_data_url(ap)\n                changed = True\n    if not changed:\n        return None\n    return json.dumps(payload).encode("utf-8")\n\n\nasync def nemotron_middleware(request: Any, call_next: Any) -> Any:\n    path = request.scope["path"].removeprefix(request.scope.get("root_path", ""))\n\n    if not _check_auth(request):\n        return JSONResponse(\n            {"error": {"message": "Unauthorized: invalid or missing Bearer token", "type": "AuthenticationError", "code": 401}},\n            status_code=HTTPStatus.UNAUTHORIZED,\n        )\n\n    if request.method == "GET" and path == "/api/ct/prompts":\n        return JSONResponse({"presets": PRESETS, "default_preset": "verbatim"})\n\n    if request.method == "GET" and path == "/api/ct/drive-list":\n        spec = request.query_params.get("path", "")\n        recursive = request.query_params.get("recursive", "1") not in ("0", "false", "False")\n        try:\n            files = await asyncio.to_thread(\n                list_drive_audio_files,\n                spec,\n                data_dir=DATA_DIR,\n                default_subdir="audio",\n                recursive=recursive,\n                mydrive_root=MYDRIVE_ROOT,\n                drive_root=DRIVE_ROOT,\n            )\n            return JSONResponse({\n                "drive_mounted": MYDRIVE_ROOT.is_dir(),\n                "query": spec,\n                "count": len(files),\n                "files": files,\n            })\n        except Exception as e:\n            return JSONResponse({"error": str(e)}, status_code=400)\n\n    if request.method == "POST" and path == "/api/ct/ytdlp-download":\n        try:\n            body = json.loads(await request.body())\n            raw_url = (body.get("url") or body.get("media_url") or body.get("audio") or "").strip()\n            if not raw_url:\n                return JSONResponse({"error": "Missing \'url\'"}, status_code=400)\n            items = await asyncio.to_thread(\n                _download_media_url_to_drive,\n                raw_url=raw_url,\n                download_dir=str(body.get("download_dir") or ""),\n                audio_format=str(body.get("audio_format") or "mp3"),\n                cookies_drive=str(body.get("cookies_drive") or ""),\n                playlist=bool(body.get("playlist", False)),\n                force_download=bool(body.get("force_download", False)),\n                download_video=bool(body.get("download_video", False)),\n                save_media_to_drive=bool(body.get("save_media_to_drive", True)),\n            )\n            return JSONResponse({"count": len(items), "files": items})\n        except ValueError as e:\n            return JSONResponse({"error": str(e)}, status_code=400)\n        except Exception as e:\n            return JSONResponse({"error": str(e)}, status_code=500)\n\n    if request.method == "POST" and path == "/api/ct/drive-transcribe":\n        try:\n            body = json.loads(await request.body())\n            code, res = await asyncio.to_thread(_run_drive_transcribe_sync, body)\n            return JSONResponse(res, status_code=code)\n        except ValueError as e:\n            return JSONResponse({"error": str(e)}, status_code=400)\n        except Exception as e:\n            return JSONResponse({"error": str(e)}, status_code=500)\n\n    if request.method == "POST" and path == "/v1/chat/completions":\n        raw_body = await request.body()\n        try:\n            rewritten = await asyncio.to_thread(_rewrite_drive_urls_in_chat_body, raw_body)\n        except ValueError as e:\n            return JSONResponse(\n                {"error": {"message": str(e), "type": "BadRequestError", "code": 400}},\n                status_code=400,\n            )\n        if rewritten is not None:\n            code, res = await asyncio.to_thread(_call_local_chat_completions, json.loads(rewritten))\n            return JSONResponse(res, status_code=code)\n        return await ctc_timestamp_middleware(request, call_next)\n\n    return await call_next(request)\n', encoding="utf-8")

GPU_MEM_UTIL = os.environ.get("CT_NEMOTRON_GPU_MEM", "0.92" if PRECISION == "bf16" else "0.85")
MAX_MODEL_LEN = int(os.environ.get("CT_NEMOTRON_MAX_MODEL_LEN", "65536"))
MAX_NUM_SEQS = int(os.environ.get("CT_NEMOTRON_MAX_SEQS", "64"))
MTP_TOKENS = int(os.environ.get("CT_NEMOTRON_MTP", "8"))
LIMIT_MM = os.environ.get(
    "CT_NEMOTRON_LIMIT_MM",
    '{"audio":{"count":6,"length":120000000}}' if PRECISION == "bf16" else '{"audio":{"count":1,"length":57600000}}',
)

VLLM_ENV = {
    "PATH": f"{VENV_DIR}/bin:/usr/local/cuda/bin:{os.environ.get('PATH', '')}",
    "PYTHONPATH": f"{NT_WORK}:{os.environ.get('PYTHONPATH', '')}",
    "VIRTUAL_ENV": str(VENV_DIR),
    "VLLM_PLUGINS": "nemo_speechlm",
    "VLLM_MAX_AUDIO_DECODE_DURATION_S": "3600",
    "VLLM_MAX_AUDIO_CLIP_FILESIZE_MB": "180",
    "VLLM_USE_V2_MODEL_RUNNER": "0",
    "FLASHINFER_WORKSPACE_BASE": str(FLASHINFER_CACHE_DIR),
    "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
    "VLLM_CACHE_ROOT": str(VLLM_CACHE_DIR),
    "TORCHINDUCTOR_CACHE_DIR": str(VLLM_CACHE_DIR / "inductor"),
    "CT_API_TOKEN": API_TOKEN,
    "CT_SERVICE_PORT": str(SERVICE_PORT),
    "CT_DATA_DIR": str(DATA_DIR),
    "CT_NEMOTRON_REPO": MODEL_REPO,
}
if PRECISION in ("fp8", "nvfp4"):
    VLLM_ENV["VLLM_DISABLED_KERNELS"] = "FlashInferFP8ScaledMMLinearKernel"

SERVER_ARGS = [
    str(VENV_DIR / "bin" / "vllm"), "serve", str(MODEL_DIR),
    "--served-model-name", MODEL_REPO, MODEL_ALIAS,
    "--host", "127.0.0.1",
    "--port", str(SERVICE_PORT),
    "--api-key", API_TOKEN,
    "--trust-remote-code",
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--max-num-batched-tokens", "4096",
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--enable-chunked-prefill",
    "--no-enable-prefix-caching",
    "--use-replayssm",
    "--mamba-backend", "flashinfer",
    "--mamba-cache-mode", "none",
    "--mm-processor-cache-gb", "0",
    "--no-async-scheduling",
    "--limit-mm-per-prompt", LIMIT_MM,
    "--reasoning-parser", "nemotron_v3",
    "--trust-request-mm-kwargs",
    "--middleware", "ct_nemotron_middleware.nemotron_middleware",
]
if PRECISION in ("fp8", "nvfp4"):
    SERVER_ARGS += ["--kv-cache-dtype", "auto"]
if MTP_TOKENS > 0:
    SERVER_ARGS += [
        "--speculative-config",
        json.dumps({"method": "mtp", "num_speculative_tokens": MTP_TOKENS}),
    ]

def start_service():
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)
    nlog(f"Starting vLLM server on 127.0.0.1:{SERVICE_PORT} (model={MODEL_REPO}, mtp={MTP_TOKENS}, ctx={MAX_MODEL_LEN})")
    return ct_spawn("service", SERVER_ARGS, env=VLLM_ENV, cwd=str(NT_WORK))

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- wait for model load/FlashInfer warmup, verify Bearer auth, supervise
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=20):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url, headers=headers, data=None if payload is None else json.dumps(payload).encode("utf-8")
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
            return resp.status, (json.loads(raw) if raw.strip().startswith(("{", "[")) else {})
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-token-probe", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 2400
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("vLLM exited during startup:\n" + ct_tail("service", 60))
    try:
        code, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 30:
        gpu_stat = ct_gpu_stat()
        last_line = (ct_tail("service", 3).strip().splitlines() or ["starting..."])[-1]
        nlog(f"Waiting for vLLM load / FlashInfer kernel warmup | GPU: {gpu_stat} | {last_line[:100]}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for vLLM server to become ready:\n" + ct_tail("service", 60))

auth_check(LOCAL_BASE, API_TOKEN)
# Exercise the audio path once so lazily compiled kernels exist before archiving caches.
import base64 as _b64, io as _io, wave as _wave
_buf = _io.BytesIO()
with _wave.open(_buf, "wb") as _wf:
    _wf.setnchannels(1); _wf.setsampwidth(2); _wf.setframerate(16000); _wf.writeframes(b"\x00\x00" * 16000)
_wcode, _ = http_json(LOCAL_BASE + "/v1/chat/completions", API_TOKEN, {
    "model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": [
        {"type": "text", "text": "Transcribe this audio."},
        {"type": "audio_url", "audio_url": {"url": "data:audio/wav;base64," + _b64.b64encode(_buf.getvalue()).decode()}},
    ]}],
    "max_tokens": 16, "temperature": 0,
}, timeout=600)
nlog(f"Warmup transcription: HTTP {_wcode}")
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        save_cache(_local_c, CACHE_ARCHIVES, CACHE_KEY, nlog)
    except (OSError, ValueError) as err:
        nlog(f"Cache archive save warning ({_local_c.name}): {err}")
code, presets_info = http_json(LOCAL_BASE + "/api/ct/prompts", API_TOKEN)
if code != 200:
    raise RuntimeError(f"Local /api/ct/prompts check failed: HTTP {code}")
nlog(f"Local endpoint verified: /v1/models (401/401/200) & /api/ct/prompts ({ len(presets_info.get('presets', {})) } presets)")

ct_public_gate(PUBLIC_URL, lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
               lambda: auth_check(PUBLIC_URL, API_TOKEN), nlog, timeout=120, interval=8)
nlog(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

ct_watchdog(lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200, start_service, nlog, dead_markers=(),
            heartbeat=lambda up: nlog(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"))
